# Вариационные автокодировщики

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Вариационные автокодировщики».

## Научная цель

Вариационный автокодировщик (VAE) учит компактное вероятностное скрытое пространство данных: близкие точки соответствуют похожим объектам. Из этого пространства можно выбирать новые точки и получать правдоподобные новые объекты, а также плавно интерполировать между существующими. Для учёного это инструмент исследования структуры данных и генерации новых кандидатов для проверки.

В этом блокноте мы обучим VAE на компактном наборе изображений цифр, построим карту двумерного скрытого пространства, сгенерируем новые изображения из скрытых точек и покажем интерполяцию между объектами. Блокнот исполняется на CPU за минуту.

## Интуиция за методом

Представьте карту города, где каждый квартал — это область похожих объектов. Обычный автокодировщик как хаотичная парковка: объекты расставлены по закоулкам без правил, между ними — «пустыри», где нет ничего осмысленного. Вариационный автокодировщик строит настоящую карту: похожие объекты лежат рядом, соседние кварталы плавно переходят друг в друга, и любая точка на карте соответствует какому-то правдоподобному объекту.

Это достигается так: вместо одного точного адреса (скрытого вектора) VAE присваивает каждому объекту небольшой «район» — нормальное распределение со средним значением и разбросом. При каждом обучающем шаге объект «живёт» в случайной точке своего района. Это заставляет соседние районы перекрываться и слипаться в единую непрерывную карту.

Для учёного это означает:
- **Генерация**: возьмите случайную точку на карте — декодировщик превратит её в правдоподобный новый объект (молекулу, сигнал, структуру материала).
- **Интерполяция**: двигайтесь по прямой между двумя точками — объект плавно трансформируется от одного к другому.
- **Анализ структуры**: посмотрите, как кластеры расположены на карте — это даст представление о смысловой близости объектов в данных.

Терминологический минимум для этого блокнота:
- **Латентное пространство** (latent space) — пространство скрытых векторов. У VAE оно непрерывное и заполненное.
- **ELBO** (Evidence Lower BOund) — «нижняя граница правдоподобия». Это функция потерь VAE: сумма ошибки восстановления и регуляризатора.
- **KL-дивергенция** (KL-divergence) — мера отличия одного распределения от другого. Здесь: насколько «район» объекта на карте отличается от стандартного нормального распределения.
- **Приём репараметризации** (reparameterization trick) — технический приём, позволяющий обучать VAE через обычный градиентный спуск, несмотря на случайность внутри сети.
- **Эпоха** — один полный проход по всем обучающим данным.
- **Функция потерь** — число, которое модель минимизирует при обучении.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch==2.3.1 scikit-learn numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор рукописных цифр `digits` из `scikit-learn` (1797 изображений 8x8). Малый размер изображений позволяет обучить VAE с двумерным скрытым пространством за минуту — а двумерность удобна для прямой визуализации скрытой карты.

In [ ]:
import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

data = load_digits()
# Каждое изображение разворачиваем в вектор длины 64, яркость в [0, 1].
X = (data.images.reshape(len(data.images), -1)
     .astype('float32') / 16.0)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
print(f'Обучающих изображений: {len(X_train)}, размер вектора: {X.shape[1]}')

## 4. Применение метода

### Шаг 1. Архитектура VAE

VAE состоит из трёх частей:

1. **Кодировщик** (`enc`) — сжимает изображение в промежуточное представление.
2. **Два линейных слоя** (`to_mu`, `to_logvar`) — выдают параметры «района» на карте: среднее (`mu`) и логарифм дисперсии (`logvar`). Логарифм используется для числовой устойчивости.
3. **Декодировщик** (`dec`) — восстанавливает изображение из точки скрытого пространства.

Функция `reparameterize` реализует **приём репараметризации**: вместо случайного вектора `z = mu + std * eps` вычисляется детерминированно, а случайность вынесена во внешний шум `eps`. Это позволяет передавать градиент через случайный шаг при обучении.

Функция `vae_loss` считает ELBO — сумму двух слагаемых:
- Ошибка восстановления (бинарная кросс-энтропия): насколько точно декодировщик восстанавливает вход.
- KL-дивергенция: насколько «район» каждого объекта на карте отличается от стандартного нормального. Это регуляризатор, заставляющий карту быть плотной и непрерывной.

Параметр `LATENT = 2` означает двумерное скрытое пространство — это удобно для прямой визуализации карты без дополнительных проекций.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

LATENT = 2          # размерность скрытого пространства (для визуализации)


class VAE(nn.Module):
    """Вариационный автокодировщик для изображений 8x8."""

    def __init__(self, n_in=64, hidden=128, latent=LATENT):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(n_in, hidden), nn.ReLU())
        self.to_mu = nn.Linear(hidden, latent)
        self.to_logvar = nn.Linear(hidden, latent)
        self.dec = nn.Sequential(
            nn.Linear(latent, hidden), nn.ReLU(),
            nn.Linear(hidden, n_in), nn.Sigmoid())

    def encode(self, x):
        h = self.enc(x)
        return self.to_mu(h), self.to_logvar(h)

    def reparameterize(self, mu, logvar):
        # Приём репараметризации: случайность вынесена в шум eps.
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.dec(z), mu, logvar


def vae_loss(recon, x, mu, logvar):
    """ELBO: ошибка восстановления + регуляризатор скрытого пространства."""
    rec = F.binary_cross_entropy(recon, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (rec + kld) / len(x)


model = VAE()
print(model)

### Шаг 2. Обучение VAE

Обучим сеть 40 эпох. На каждой эпохе оптимизируется ELBO-потеря: она должна убывать по мере того, как модель учится и восстанавливать объекты, и строить плотную карту скрытого пространства.

Числа, выводимые в процессе, — это ELBO-потеря на обучающей выборке. Они убывают в первых эпохах, а затем стабилизируются: это нормально.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

ds = TensorDataset(torch.from_numpy(X_train))
loader = DataLoader(ds, batch_size=64, shuffle=True)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

history = []
for epoch in range(1, 41):
    model.train()
    ep = 0.0
    for (xb,) in loader:
        opt.zero_grad()
        recon, mu, logvar = model(xb)
        loss = vae_loss(recon, xb, mu, logvar)
        loss.backward()
        opt.step()
        ep += loss.item() * len(xb)
    history.append(ep / len(X_train))
    if epoch % 10 == 0:
        print(f'Эпоха {epoch:2d}: ELBO-потеря {history[-1]:.2f}')

### Шаг 3. Карта скрытого пространства и генерация

Построим три ключевые визуализации, которые показывают, чему научился VAE.

**Визуализация 1 — карта скрытого пространства:** передадим тестовые изображения через кодировщик и возьмём средние значения (`mu`) их «районов» на карте. Двумерность позволяет нанести точки прямо на плоскость.

**Визуализация 2 — сетка генерации:** равномерно разобьём скрытое пространство на сетку и для каждого узла попросим декодировщик создать изображение. Это позволяет «пройти» по всей карте и увидеть, как плавно меняются генерируемые объекты.

**Визуализация 3 — интерполяция:** возьмём два конкретных объекта, найдём их положения на карте и шагаем по прямой от одного к другому, генерируя промежуточные объекты.

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    mu_test, _ = model.encode(torch.from_numpy(X_test))
mu_test = mu_test.numpy()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.6))

# Карта скрытого пространства: цвет — класс цифры.
sc = axes[0].scatter(mu_test[:, 0], mu_test[:, 1], c=y_test,
                     cmap='tab10', s=22, alpha=0.8,
                     edgecolor='white', linewidth=0.3)
axes[0].set_title('Скрытое пространство VAE')
axes[0].set_xlabel('Скрытая ось 1')
axes[0].set_ylabel('Скрытая ось 2')
fig.colorbar(sc, ax=axes[0], fraction=0.046, pad=0.04,
             label='Класс цифры')

# Сетка изображений, сгенерированных из узлов скрытого пространства.
grid_n = 12
lo, hi = mu_test.min(), mu_test.max()
gx = np.linspace(lo, hi, grid_n)
gy = np.linspace(hi, lo, grid_n)
canvas = np.zeros((grid_n * 8, grid_n * 8))
with torch.no_grad():
    for i, zy in enumerate(gy):
        for j, zx in enumerate(gx):
            z = torch.tensor([[zx, zy]], dtype=torch.float32)
            img = model.dec(z).numpy().reshape(8, 8)
            canvas[i * 8:(i + 1) * 8, j * 8:(j + 1) * 8] = img
axes[1].imshow(canvas, cmap='gray_r')
axes[1].set_title('Генерация из скрытого пространства')
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].grid(False)

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Левый — карта скрытого пространства:** каждая точка — тестовое изображение, цвет — класс цифры. У хорошо обученного VAE одинаковые цифры собираются в компактные области, а похожие цифры (например, 3 и 8) расположены рядом. Всё пространство заполнено без больших «пустот».
- **Правый — сетка генерации:** каждое маленькое изображение создано из соответствующей точки карты. Плавные переходы между соседними изображениями подтверждают, что скрытое пространство непрерывно. Резкие скачки или бессмысленные изображения — признак «пустот» в пространстве или недостаточного обучения.

In [ ]:
# Интерполяция: плавный переход между двумя объектами в скрытом пространстве.
model.eval()
with torch.no_grad():
    mu_a, _ = model.encode(torch.from_numpy(X_test[0:1]))
    mu_b, _ = model.encode(torch.from_numpy(X_test[1:2]))

steps = 9
fig, axes = plt.subplots(1, steps, figsize=(12, 1.9))
for k, alpha in enumerate(np.linspace(0, 1, steps)):
    z = (1 - alpha) * mu_a + alpha * mu_b
    with torch.no_grad():
        img = model.dec(z).numpy().reshape(8, 8)
    axes[k].imshow(img, cmap='gray_r')
    axes[k].set_xticks([]); axes[k].set_yticks([])
    axes[k].grid(False)
fig.suptitle('Интерполяция в скрытом пространстве (плавный переход между объектами)', fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график (интерполяция):**

Девять изображений расположены слева направо: крайнее левое — первый исходный объект, крайнее правое — второй. Промежуточные изображения получены из точек, равномерно расставленных по прямой между ними в скрытом пространстве.

Если переход плавный — латентное пространство непрерывно и структурировано: нет резких «провалов», где декодировщик выдаёт бессмыслицу. Это ключевое отличие VAE от обычного автокодировщика. В научном применении такой путь по карте соответствует, например, постепенному изменению молекулы от одной структуры к другой.

## 5. Интерпретация результата

- **Карта скрытого пространства**: объекты одного класса собираются в области, похожие классы лежат рядом. Это и есть структурированное скрытое пространство, которому учит VAE.
- **Сетка генерации**: каждое изображение получено из своей точки скрытого пространства. Плавные переходы между соседними клетками подтверждают непрерывность пространства.
- **Интерполяция**: при движении по прямой между двумя объектами изображение меняется плавно — это отличает VAE от обычного автокодировщика.
- Выборки VAE обычно **более размытые**, чем у состязательных и диффузионных моделей; в «дырах» скрытого пространства декодировщик может порождать недостоверные объекты.

Слишком сильный вес регуляризатора ведёт к коллапсу апостериора — декодировщик начинает игнорировать скрытый код.

## 6. Попробуйте на своих данных

VAE применим к непрерывным данным: изображениям, спектрам, профилям сигналов.

1. Подготовьте матрицу объектов `X` формы `(наблюдение, признак)` со значениями, отмасштабированными в диапазон [0, 1].
2. Снимите комментарии в ячейке ниже и подставьте свои данные.
3. Размерность скрытого пространства `LATENT` = 2 удобна для визуализации; для генерации обычно берут больше (8–32).
4. Для дискретных данных (молекулярные графы, строки) нужны специальные варианты VAE.

## 7. Поэкспериментируйте сами

Запустите заново раздел 4, изменив один из параметров ниже:

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `LATENT = 4` | Увеличить до 4 | Карта станет 4D — потребуется PCA для отображения; качество генерации улучшится |
| `hidden = 64` | Уменьшить скрытый слой до 64 | Обучение быстрее, но восстановление хуже — ищем компромисс |
| `n_epochs = 80` | Удвоить число эпох | Стабилизируется ли карта? Улучшается ли сетка генерации? |
| `lr = 5e-4` | Снизить скорость обучения | Медленнее, но иногда более структурированная карта |
| `LATENT = 2`, подать на сетку другой диапазон | Вместо `[lo, hi]` взять `[-3, 3]` | Что находится за краями карты? Обычно бессмыслица — это «дыры» латентного пространства |

Главное наблюдение: при малом `LATENT` карта хорошо организована, но генерация размыта; при большом `LATENT` — наоборот. Это фундаментальный компромисс VAE.

In [ ]:
# --- Шаблон загрузки своих данных ---
# import pandas as pd
#
# X = pd.read_csv('my_data.csv').to_numpy(dtype='float32')
# # Масштабирование каждого признака в [0, 1]:
# X = (X - X.min(0)) / (X.max(0) - X.min(0) + 1e-8)
#
# print(f'Объектов: {len(X)}, признаков: {X.shape[1]}')
# При смене размерности входа задайте n_in в конструкторе VAE.

## 8. Краткие выводы

- VAE — это улучшенный автокодировщик, который строит непрерывное вероятностное скрытое пространство: каждый объект кодируется не в точку, а в «район» (нормальное распределение).
- Функция потерь ELBO балансирует два требования: точно восстанавливать объекты (ошибка восстановления) и держать карту плотной без «пустот» (KL-дивергенция).
- Приём репараметризации позволяет обучать сеть через обычный градиентный спуск, несмотря на стохастику.
- Двумерное скрытое пространство удобно для визуализации; для реальных задач генерации обычно используют 8–32 измерения.
- Интерполяция между объектами возможна только в непрерывном пространстве — и это главное практическое отличие VAE от обычного автокодировщика.
- Выборки VAE несколько размытее, чем у GAN или диффузионных моделей; если важна чёткость — рассмотрите их. Если важна скорость и стабильность обучения — VAE предпочтительнее.